# MiniGPT：从文本到 logits 和 loss

本实验使用极小随机模型验证接口与因果性，不追求生成质量。

In [ ]:
from about_llm.from_scratch import ByteTokenizer

tokenizer = ByteTokenizer()
text = 'LLM 你好'
token_ids = tokenizer.encode(text)
assert tokenizer.decode(token_ids) == text
print('UTF-8 bytes:', token_ids)


In [ ]:
import torch

from about_llm.from_scratch.gpt_torch import GPTConfig, MiniGPT

torch.manual_seed(0)
config = GPTConfig(
    vocab_size=256, context_length=16, model_dim=32,
    num_heads=4, num_layers=2, mlp_ratio=2,
)
model = MiniGPT(config).eval()
tokens = torch.tensor([token_ids[:8]], dtype=torch.long)
inputs, targets = tokens[:, :-1], tokens[:, 1:]
logits, loss = model(inputs, targets)
print('PyTorch logits:', tuple(logits.shape), 'loss:', round(loss.item(), 4))
assert logits.shape == (*inputs.shape, config.vocab_size)


In [ ]:
first = torch.tensor([[1, 2, 3, 4]])
second = torch.tensor([[1, 2, 9, 10]])
first_logits, _ = model(first)
second_logits, _ = model(second)
torch.testing.assert_close(first_logits[:, :2], second_logits[:, :2])
print('Causality check passed: future tokens do not change past logits.')


In [ ]:
import jax
import jax.numpy as jnp

from about_llm.from_scratch.gpt_jax import JAXGPTConfig, cross_entropy_loss, forward, init_params

jax_config = JAXGPTConfig(
    vocab_size=256, context_length=16, model_dim=32,
    num_heads=4, num_layers=2, mlp_ratio=2,
)
params = init_params(jax.random.key(0), jax_config)
jax_logits = forward(params, jnp.asarray(inputs.numpy()), jax_config)
jax_loss = cross_entropy_loss(jax_logits, jnp.asarray(targets.numpy()))
print('JAX logits:', jax_logits.shape, 'loss:', round(float(jax_loss), 4))
assert jax_logits.shape == (*inputs.shape, jax_config.vocab_size)


## 下一步

随机初始化模型只验证结构。训练实验会加入数据 batch、optimizer、梯度裁剪、验证 loss、checkpoint 和生成对比。